# P04 — Prévision de la consommation électrique nationale (éco2mix)

**Contexte métier.** RTE prévoit chaque jour à J-1 la consommation nationale pour ajuster le parc de production et garantir l'équilibre offre/demande. Un modèle étudiant peut challenger la prévision officielle. Enjeu : ~52 GW de consommation moyenne, pics hivernaux à 96 GW.

**Objectif de l'EDA.** Établir la cible (consommation demi-horaire), la baseline à battre (prévision J-1 de RTE, MAE 1 065 MW) et les régresseurs temporels (heure, saison, lag 96 = veille même heure).

**Données.** éco2mix national temps réel — RTE/Réseau de transport d'électricité via ODRÉ (odre.opendatasoft.com). Licence ouverte Etalab 2.0. 151 917 points demi-horaires de 2018 à août 2026.

## 1. Téléchargement

Export CSV de l'API ODRÉ (OpenDataSoft) filtré à partir de 2018 :

In [ ]:
import urllib.request, os
os.makedirs("data", exist_ok=True)

URL = ("https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets/eco2mix-national-tr/exports/csv"
       "?where=" + urllib.parse.quote("date_heure >= '2018-01-01'") + "&use_labels=true")
if not os.path.exists("data/eco2mix-national-cons-2018plus.csv"):
    urllib.request.urlretrieve(URL, "data/eco2mix-national-cons-2018plus.csv")
print(os.path.getsize("data/eco2mix-national-cons-2018plus.csv") / 1e6, "Mo")

## 2. Chargement

Séparateur `;`, décimales à virgule. Parse de la date et création des variables temporelles (heure, mois, jour) et du lag 96 points (24 h) :

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("data/eco2mix-national-cons-2018plus.csv", sep=";", parse_dates=["Date et Heure"])
df.columns = [c.strip() for c in df.columns]
df = df.rename(columns={"Consommation (MW)": "conso", "Prévision J-1 (MW)": "prev_j1"})
df = df.dropna(subset=["conso"]).sort_values("Date et Heure").reset_index(drop=True)
df["heure"] = df["Date et Heure"].dt.hour
df["mois"] = df["Date et Heure"].dt.month
df["jour"] = df["Date et Heure"].dt.dayofweek
df["conso_lag_96"] = df["conso"].shift(96)   # même heure, veille
print(len(df), "points,", df["Date et Heure"].min(), "→", df["Date et Heure"].max())

## 3. Analyse

Saisonnalité annuelle et journalière, corrélation avec la prévision J-1 et le lag 96, et calcul de la baseline : MAE de la prévision RTE contre MAE naïve (conso de la veille).

In [ ]:
print(df["conso"].describe().round(0).to_dict())
print()
print("Conso moyenne par mois :", df.groupby("mois")["conso"].mean().round(0).to_dict())
print("Conso moyenne par heure :", df.groupby("heure")["conso"].mean().round(0).to_dict())
print("Conso moyenne par jour (0=lundi) :", df.groupby("jour")["conso"].mean().round(0).to_dict())
print()
c = df[["conso", "conso_lag_96", "prev_j1"]].corr("spearman")
print("Corrélations Spearman :", c["conso"].round(2).to_dict())
mae_naif = (df["conso"] - df["conso_lag_96"]).abs().mean()
mae_rte = (df["conso"] - df["prev_j1"]).abs().mean()
print("MAE naïf (veille) :", round(mae_naif), "MW — MAE prévision J-1 RTE :", round(mae_rte), "MW")

## 4. Résultats observés (exécution du 16/10/2026)

- n lignes : 151917
- periode : 2018-01-01 00:00:00+00:00 → 2026-08-31 21:30:00+00:00
- conso stats : count : 151 917.00, mean : 51 690.00, std : 11 177.00, min : 29 124.00, 5% : 35 890.00, 25% : 43 357.00, 50% : 49 922…
- conso moyenne par mois : 1 : 67 381.00, 2 : 63 370.00, 3 : 56 696.00, 4 : 48 654.00, 5 : 43 918.00, 6 : 43 994.00, 7 : 44 685.00, 8 : 42 075.0…
- conso moyenne par heure : 0 : 47 195.00, 1 : 45 598.00, 2 : 43 930.00, 3 : 43 558.00, 4 : 45 087.00, 5 : 48 241.00, 6 : 51 702.00, 7 : 54 156.0…
- conso moyenne par jour semaine : 0 : 52 606.00, 1 : 53 680.00, 2 : 53 711.00, 3 : 53 533.00, 4 : 52 989.00, 5 : 48 549.00, 6 : 46 757.00
- corr spearman : conso : 1.00, conso_lag_96 : 0.85, prev_j1 : 0.99, nucleaire : 0.68, eolien : 0.20, gaz : 0.66, solaire : 0.05
- mae naif hier mw : 4 373.00
- mae prev j1 mw : 1 065.00

## 5. Variable cible et régresseurs

**Cible (régression / prévision de série temporelle) : `conso` — consommation nationale en MW, demi-horaire.**

**Baseline à battre : MAE 4 373 MW** pour le naïf (conso de la veille même heure) et **MAE 1 065 MW** pour la prévision J-1 de RTE. Un modèle étudiant visant < 2 000 MW est déjà performant.

**Régresseurs retenus :**
- `conso_lag_96` (veille même heure, corr. 0,85) et autres lags horaires
- `heure`, `mois`, `jour` : deux saisonnalités très nettes (67 GW en janvier, 44 GW en août ; pic 18 h-20 h ; semaine > week-end)
- Filières de production (`nucléaire` corr. 0,68, `gaz`) — mais attention : elles sont déterminées par la consommation, à utiliser avec prudence
- Température externe (à joindre : Météo-France) pour capter le thermosensibilité

**Pièges :**
- Ne jamais entraîner sur le futur : split temporel strict (train < test dans le temps), pas de split aléatoire
- `prev_j1` est la cible d'un modèle concurrent : elle sert d'évaluation, pas de feature
- Passages DST et trous dans le flux : le lag 96 doit être contrôlé avant usage
- La consommation a changé de niveau avec la crise énergétique 2022 : tester la robustesse sur plusieurs années